In [ ]:
import csv
import os
import pyshark
from datetime import datetime

In [ ]:
def extract_modbus_to_csv(pcap_file, output_csv='modbus_data.csv'):
    """
    Extract Modbus/TCP packets from a pcap file and save to CSV.
    
    Args:
        pcap_file: Path to the pcap file
        output_csv: Output CSV filename
    """
    # Open the pcap file
    print(f"Opening pcap file: {pcap_file}")
    cap = pyshark.FileCapture(pcap_file, display_filter='modbus')
    
    # Define CSV headers for Modbus/TCP fields
    headers = [
        'timestamp',
        'src_ip',
        'dst_ip',
        'src_port',
        'dst_port',
        'transaction_id',
        'protocol_id',
        'length',
        'unit_id',
        'function_code',
        'function_name',
        'reference_number',
        'word_count',
        'byte_count',
        'data'
    ]
    
    modbus_packets = []
    packet_count = 0
    
    print("Extracting Modbus/TCP packets...")
    
    try:
        for packet in cap:
            packet_count += 1
            
            # Extract packet information
            packet_data = {}
            
            # Timestamp
            packet_data['timestamp'] = packet.sniff_timestamp
            
            # IP layer
            if hasattr(packet, 'ip'):
                packet_data['src_ip'] = packet.ip.src
                packet_data['dst_ip'] = packet.ip.dst
            else:
                packet_data['src_ip'] = ''
                packet_data['dst_ip'] = ''
            
            # TCP layer
            if hasattr(packet, 'tcp'):
                packet_data['src_port'] = packet.tcp.srcport
                packet_data['dst_port'] = packet.tcp.dstport
            else:
                packet_data['src_port'] = ''
                packet_data['dst_port'] = ''
            
            # Modbus layer
            if hasattr(packet, 'modbus'):
                modbus = packet.modbus
                
                # Transaction ID
                packet_data['transaction_id'] = getattr(modbus, 'transid', '')
                
                # Protocol ID (should be 0 for Modbus)
                packet_data['protocol_id'] = getattr(modbus, 'protid', '')
                
                # Length
                packet_data['length'] = getattr(modbus, 'len', '')
                
                # Unit ID
                packet_data['unit_id'] = getattr(modbus, 'unitid', '')
                
                # Function Code
                packet_data['function_code'] = getattr(modbus, 'func_code', '')
                
                # Function Name (if available)
                packet_data['function_name'] = getattr(modbus, 'func_code_name', '')
                
                # Reference Number (starting address)
                packet_data['reference_number'] = getattr(modbus, 'reference_num', '')
                
                # Word Count
                packet_data['word_count'] = getattr(modbus, 'word_cnt', '')
                
                # Byte Count
                packet_data['byte_count'] = getattr(modbus, 'byte_cnt', '')
                
                # Extract data fields (registers, coils, etc.)
                data_fields = []
                for field_name in dir(modbus):
                    if 'data' in field_name.lower() or 'register' in field_name.lower():
                        try:
                            value = getattr(modbus, field_name)
                            if value and not callable(value):
                                data_fields.append(f"{field_name}={value}")
                        except:
                            pass
                
                packet_data['data'] = '; '.join(data_fields) if data_fields else ''
            else:
                # Fill with empty values if no Modbus layer
                for key in headers[5:]:
                    packet_data[key] = ''
            
            modbus_packets.append(packet_data)
            
            if packet_count % 100 == 0:
                print(f"Processed {packet_count} packets...")
    
    except Exception as e:
        print(f"Finished processing. Total packets: {packet_count}")
        print(f"Note: {e}")
    
    finally:
        cap.close()
    
    # Write to CSV
    print(f"\nWriting {len(modbus_packets)} Modbus packets to {output_csv}")
    
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        writer.writerows(modbus_packets)
    
    print(f"✓ Successfully extracted {len(modbus_packets)} Modbus/TCP packets to {output_csv}")
    
    return len(modbus_packets)